# Regularized Regression: Ridge, Lasso, Elastic Net Comparison

This comprehensive notebook compares three regularization techniques for linear regression.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.linear_model import LinearRegression, Ridge, Lasso, ElasticNet
from sklearn.datasets import load_diabetes
from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import mean_squared_error, r2_score, mean_absolute_error

np.random.seed(42)

In [ ]:
# Use diabetes dataset with many features
X, y = load_diabetes(return_X_y=True)
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

# Standardize
scaler = StandardScaler()
X_train = scaler.fit_transform(X_train)
X_test = scaler.transform(X_test)

print(f"Train: {X_train.shape}, Test: {X_test.shape}")
print(f"Features: {X.shape[1]}")

In [ ]:
# Models with regularization
models = {
    'Linear Regression': LinearRegression(),
    'Ridge (α=1.0)': Ridge(alpha=1.0),
    'Lasso (α=0.1)': Lasso(alpha=0.1, max_iter=10000),
    'Elastic Net (α=0.1, l1_ratio=0.5)': ElasticNet(alpha=0.1, l1_ratio=0.5, max_iter=10000)
}

results = {}

for name, model in models.items():
    model.fit(X_train, y_train)
    y_pred_train = model.predict(X_train)
    y_pred_test = model.predict(X_test)
    
    train_r2 = r2_score(y_train, y_pred_train)
    test_r2 = r2_score(y_test, y_pred_test)
    test_mse = mean_squared_error(y_test, y_pred_test)
    test_mae = mean_absolute_error(y_test, y_pred_test)
    
    results[name] = {'Train R²': train_r2, 'Test R²': test_r2, 'Test MSE': test_mse, 'Test MAE': test_mae}
    
    # Coefficient magnitudes
    coef_magnitude = np.sum(np.abs(model.coef_))
    n_nonzero = np.sum(model.coef_ != 0)
    
    print(f"{name}:")
    print(f"  Train R²: {train_r2:.4f}, Test R²: {test_r2:.4f}")
    print(f"  Coefficient magnitude: {coef_magnitude:.2f}, Non-zero: {n_nonzero}")
    print()

In [ ]:
df_results = pd.DataFrame(results).T
print("\n=== REGULARIZATION COMPARISON ===")
print(df_results.round(4))

In [ ]:
fig, ax = plt.subplots(figsize=(12, 6))

for name, model in models.items():
    coefs = model.coef_ if hasattr(model, 'coef_') else model.coef_
    ax.plot(range(len(coefs)), coefs, 'o-', label=name, alpha=0.7)

ax.set_xlabel('Feature Index')
ax.set_ylabel('Coefficient Value')
ax.set_title('Coefficient Comparison Across Regularization Methods')
ax.legend()
ax.grid(alpha=0.3)
plt.tight_layout()
plt.show()

In [ ]:
print("""
Regularization Methods Summary:

Ridge (L2):
- Shrinks coefficients uniformly
- Keeps all features (never zeros)
- Good for multicollinearity

Lasso (L1):
- Performs feature selection (zeros out coefficients)
- Sparse solutions
- Interpretable models

Elastic Net (L1 + L2):
- Combines Ridge and Lasso benefits
- Feature selection + grouping effect
- Handles correlated features better than Lasso
""")